# 02 — Predição, desempenho e explicabilidade em classificação

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flavioluizseixas/aprendizado-de-maquina-para-saude/blob/main/notebooks/02_aprendizado_supervisionado.ipynb)

**Duração estimada:** 90–110 minutos  
**Pré-requisitos:** Notebook 01 ou noções de pandas e classificação binária.

## Objetivos

- construir um pipeline sem vazamento
- avaliar regressão logística e Random Forest somente no teste reservado
- interpretar sensibilidade, especificidade, ROC-AUC e PR-AUC
- interpretar odds ratios ajustadas com IC95% e SHAP na regressão logística
- investigar uma Random Forest separadamente com permutação e SHAP

## Fonte e licença

[CDC Diabetes Health Indicators — descrição das variáveis](https://archive.ics.uci.edu/dataset/891/cdc+diabetes+health+indicators), conjunto 891 da UCI, derivado do BRFSS. Consulte a tabela de variáveis para interpretar os códigos dos atributos, como 0 e 1.

Fonte UCI sob CC BY 4.0; cite o conjunto e sua publicação.

> **Uso responsável:** Este material tem finalidade exclusivamente educacional. Os resultados não devem ser usados para diagnóstico, prognóstico, tratamento, gestão assistencial ou decisão de saúde pública sem validação adequada, análise de contexto e supervisão de profissionais qualificados.

## Onde executar

### Google Colab

Use o botão **Open In Colab** no início do notebook e escolha **Executar tudo**. A célula de preparação clona ou atualiza o repositório em `/content`, instala somente as dependências ausentes e fixa a semente aleatória.

### Computador local

Requisitos: Git e Python 3.10–3.13. No terminal, clone o projeto e crie um ambiente virtual:

```bash
git clone https://github.com/flavioluizseixas/aprendizado-de-maquina-para-saude.git
cd aprendizado-de-maquina-para-saude
python -m venv .venv
```

Ative-o no Windows PowerShell com `.\.venv\Scripts\Activate.ps1` ou, no Linux/macOS, com `source .venv/bin/activate`.

Instale somente as dependências deste encontro e abra o notebook a partir da raiz do repositório:

```bash
python -m pip install -e ".[supervised]"
jupyter lab notebooks/02_aprendizado_supervisionado.ipynb
```

Não é necessário alterar caminhos nem fazer upload de arquivos. Fora do Colab, a próxima célula usa o repositório local e o mesmo ambiente Python selecionado como kernel do Jupyter.

## Preparação do ambiente

> Como fixar dependências e semente?

In [ ]:
# Preparação reproduzível do ambiente (a instalação ocorre só se faltar pacote).
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

REPO = "flavioluizseixas/aprendizado-de-maquina-para-saude"
REPO_DIR = Path("/content") / REPO.split("/")[-1]
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    command = ["git", "clone", f"https://github.com/{REPO}.git", str(REPO_DIR)]
    if REPO_DIR.exists():
        command = ["git", "-C", str(REPO_DIR), "pull", "--ff-only"]
    subprocess.run(command, check=True)
    os.chdir(REPO_DIR)
else:
    candidates = [Path.cwd(), Path.cwd().parent]
    project = next((p for p in candidates if (p / "src").exists()), Path.cwd())
    os.chdir(project)

packages = {'numpy': 'numpy>=1.26,<3', 'pandas': 'pandas>=2.1,<4', 'matplotlib': 'matplotlib>=3.8,<4', 'seaborn': 'seaborn>=0.13,<1', 'sklearn': 'scikit-learn>=1.4,<2', 'requests': 'requests>=2.31,<3', 'ucimlrepo': 'ucimlrepo>=0.0.7,<1', 'shap': 'shap>=0.45,<1', 'statsmodels': 'statsmodels>=0.14,<1'}
missing = [spec for module, spec in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

from src.config import RANDOM_STATE, seed_everything
seed_everything(RANDOM_STATE)
print(f"Ambiente pronto em {Path.cwd()} | Colab={IN_COLAB} | semente={RANDOM_STATE}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import shap
import statsmodels.api as sm
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from src.data_loading import load_cdc_diabetes
from src.evaluation import classification_report_health, plot_classification_curves, report_frame

In [ ]:
FAST_MODE = True
SAMPLE_SIZE = 30_000 if FAST_MODE else 40_000

TARGET_NAME = "Indicador de diabetes"
target_labels = {0: "0 Sem diabetes", 1: "1 Pré-diabetes/diabetes"}
feature_labels = {
    "HighBP": "Pressão alta",
    "HighChol": "Colesterol alto",
    "CholCheck": "Verificação de colesterol (5 anos)",
    "BMI": "IMC",
    "Smoker": "Fumou ao menos 100 cigarros",
    "Stroke": "Histórico de AVC",
    "HeartDiseaseorAttack": "Doença cardíaca ou infarto",
    "PhysActivity": "Atividade física (30 dias)",
    "Fruits": "Consumo diário de frutas",
    "Veggies": "Consumo diário de vegetais",
    "HvyAlcoholConsump": "Consumo elevado de álcool",
    "AnyHealthcare": "Cobertura de saúde",
    "NoDocbcCost": "Sem consulta por custo",
    "GenHlth": "Saúde geral (1 excelente–5 ruim)",
    "MentHlth": "Saúde mental ruim (dias/30)",
    "PhysHlth": "Saúde física ruim (dias/30)",
    "DiffWalk": "Dificuldade para caminhar",
    "Sex": "Sexo",
    "Age": "Faixa etária",
    "Education": "Escolaridade",
    "Income": "Faixa de renda",
}

data, metadata = load_cdc_diabetes(SAMPLE_SIZE, random_state=RANDOM_STATE)
X = data.drop(columns="Diabetes_binary").rename(columns=feature_labels)
y = data["Diabetes_binary"].astype(int)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print("Treino:", X_train.shape, "| teste reservado:", X_test.shape)

### Dicionário de variáveis e rótulos

A classe positiva (`1`) reúne **pré-diabetes ou diabetes**; portanto, o modelo não separa essas duas condições. Nos atributos binários, `0` e `1` representam categorias documentadas na fonte, não intensidades. Variáveis como saúde geral, idade, escolaridade e renda usam códigos ordinais.

A tabela seguinte vem diretamente dos metadados da UCI. Consulte também a [página da base](https://archive.ics.uci.edu/dataset/891/cdc+diabetes+health+indicators) antes de interpretar qualquer atributo.

In [ ]:
coding_summary = pd.DataFrame({
    "Grupo": [
        "Desfecho", "Indicadores binários", "Sexo", "Saúde geral", "Demográficos ordinais"
    ],
    "Como interpretar": [
        "0=sem diabetes; 1=pré-diabetes ou diabetes",
        "0=não/ausente; 1=sim/presente (exceto a variável Sex)",
        "Sex: 0=feminino; 1=masculino",
        "1=excelente; 2=muito boa; 3=boa; 4=regular; 5=ruim",
        "Age, Education e Income são faixas ordenadas; consulte os níveis na fonte",
    ],
})
display(coding_summary.style.hide(axis="index"))

variable_dictionary = metadata.get("variables")
if isinstance(variable_dictionary, pd.DataFrame):
    dictionary_columns = [
        column for column in ["name", "role", "type", "description", "units"]
        if column in variable_dictionary.columns
    ]
    variable_dictionary = variable_dictionary[dictionary_columns].copy()
    translated_names = {**feature_labels, "Diabetes_binary": TARGET_NAME}
    if "name" in variable_dictionary:
        variable_dictionary.insert(
            1, "nome_no_notebook",
            variable_dictionary["name"].map(translated_names).fillna(variable_dictionary["name"]),
        )
    display(variable_dictionary.rename(columns={
        "name": "nome_original", "role": "papel", "type": "tipo",
        "description": "descrição_UCI", "units": "unidade",
    }))

class_distribution = pd.DataFrame({
    "n": y.value_counts().sort_index(),
    "percentual": 100 * y.value_counts(normalize=True).sort_index(),
})
class_distribution.index = class_distribution.index.map(target_labels)
display(class_distribution.round(1).rename_axis("classe do desfecho"))

## Pergunta orientadora

> Um modelo consegue ordenar pessoas com e sem o indicador-alvo, e quais erros aparecem quando escolhemos um limiar?

## Inspeção de correlações

> Quais associações monotônicas merecem atenção antes do modelo?

In [ ]:
modeling_data = X.assign(**{TARGET_NAME: y})
corr = modeling_data.corr(method="spearman", numeric_only=True)
target_corr = corr[TARGET_NAME].drop(TARGET_NAME)
top_corr = target_corr.abs().sort_values(ascending=False).head(12).index
display(target_corr.loc[top_corr].sort_values(key=abs, ascending=False).rename("Spearman").to_frame())
plt.figure(figsize=(10, 7))
sns.heatmap(corr.loc[[*top_corr, TARGET_NAME], [*top_corr, TARGET_NAME]], cmap="vlag", center=0)
plt.title("Correlação de Spearman entre os principais atributos")
plt.show()

### Como interpretar

Spearman resume associação monotônica. Códigos ordinais, não linearidade e variáveis omitidas afetam a leitura; correlação não demonstra causalidade.

## Modelo preditivo de regressão logística

> Como ajustar o pré-processamento somente no treinamento e avaliar apenas no teste reservado?

O conjunto de treinamento serve para ajustar o classificador. Nenhuma medida de desempenho de re-substituição no treinamento será exibida, pois ela tende a ser otimista. Todas as métricas seguintes serão calculadas exclusivamente no teste.

In [ ]:
pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1_000, class_weight="balanced", random_state=RANDOM_STATE)),
])

## Avaliação da regressão logística no teste

> O que acontece no conjunto que não participou do ajuste?

Para positivos (VP), negativos (VN), falsos positivos (FP) e falsos negativos (FN):

- sensibilidade = VP / (VP + FN);
- especificidade = VN / (VN + FP);
- precisão = VP / (VP + FP);
- F1 = 2 × precisão × sensibilidade / (precisão + sensibilidade).

In [ ]:
pipeline.fit(X_train, y_train)
y_prob = pipeline.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= 0.5).astype(int)
report = classification_report_health(y_test, y_pred, y_prob)
display(report_frame(report).round(3))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=[target_labels[0], target_labels[1]], cmap="Blues",
)
plt.title("Matriz de confusão — teste")
plt.show()
plot_classification_curves(y_test, y_prob); plt.show()

### Como interpretar

Um falso negativo é um positivo no conjunto que o limiar não sinalizou; um falso positivo é um negativo sinalizado. O custo de cada erro depende do contexto — não é decidido pela ROC-AUC. Alterar 0,5 muda esse equilíbrio.

## Explicabilidade da regressão logística

> Quais associações condicionais o modelo estima e como elas aparecem em uma previsão individual?

O pipeline anterior tem finalidade **preditiva**: usa regularização, pesos de classe e padronização. Esses recursos são úteis para previsão, mas seus coeficientes não devem ser apresentados como estimativas epidemiológicas clássicas.

Para estimar *odds ratios* ajustadas, ajustaremos um segundo modelo logístico, agora inferencial:

- sem `class_weight` e sem penalização;
- variáveis binárias mantidas como 0/1, com 0 como referência;
- IMC e dias de saúde ruim mantidos nas unidades originais;
- saúde geral, idade, escolaridade e renda tratadas como categorias, sem supor distâncias iguais entre os níveis;
- nível 1 usado como referência nas variáveis categóricas;
- IC95% robustos HC3 para representar a incerteza amostral sob as hipóteses declaradas.

Como o objetivo aqui é estimar associações, e não medir generalização preditiva, o modelo usa toda a amostra analítica. Cada OR é ajustada simultaneamente pelas demais variáveis do quadro.

In [ ]:
binary_features = [
    "Pressão alta", "Colesterol alto", "Verificação de colesterol (5 anos)",
    "Fumou ao menos 100 cigarros", "Histórico de AVC",
    "Doença cardíaca ou infarto", "Atividade física (30 dias)",
    "Consumo diário de frutas", "Consumo diário de vegetais",
    "Consumo elevado de álcool", "Cobertura de saúde", "Sem consulta por custo",
    "Dificuldade para caminhar", "Sexo",
]
continuous_features = [
    "IMC", "Saúde mental ruim (dias/30)", "Saúde física ruim (dias/30)",
]
category_levels = {
    "Saúde geral (1 excelente–5 ruim)": list(range(1, 6)),
    "Faixa etária": list(range(1, 14)),
    "Escolaridade": list(range(1, 7)),
    "Faixa de renda": list(range(1, 9)),
}

epidemiology_data = X.assign(**{TARGET_NAME: y}).dropna().copy()
print(
    f"Amostra analítica completa: {len(epidemiology_data):,} de {len(X):,} participantes; "
    f"eventos: {epidemiology_data[TARGET_NAME].sum():,}."
)

epi_predictors = epidemiology_data[continuous_features + binary_features].astype(float).copy()
for feature, levels in category_levels.items():
    observed = set(epidemiology_data[feature].astype(int).unique())
    unexpected = observed.difference(levels)
    if unexpected:
        raise ValueError(f"Códigos inesperados em {feature}: {sorted(unexpected)}")
    coded = pd.Series(
        pd.Categorical(
            epidemiology_data[feature].astype(int),
            categories=levels,
            ordered=True,
        ),
        index=epidemiology_data.index,
        name=feature,
    )
    dummies = pd.get_dummies(
        coded, prefix=feature, prefix_sep=" = ", drop_first=True, dtype=float
    )
    epi_predictors = pd.concat([epi_predictors, dummies], axis=1)

epi_design = sm.add_constant(epi_predictors, has_constant="add").astype(float)
epi_outcome = epidemiology_data[TARGET_NAME].astype(int)
epi_result = sm.GLM(
    epi_outcome, epi_design, family=sm.families.Binomial()
).fit(cov_type="HC3")

references = pd.DataFrame({
    "variáveis": [
        "Binárias (exceto Sexo)", "Sexo", "Saúde geral",
        "Faixa etária", "Escolaridade", "Faixa de renda", "Contínuas",
    ],
    "contraste da OR": [
        "1 (sim/presente) versus 0 (não/ausente)",
        "1 (masculino) versus 0 (feminino)",
        "cada nível versus 1 (excelente)",
        "cada faixa versus faixa 1", "cada nível versus nível 1",
        "cada faixa versus faixa 1", "aumento de uma unidade original",
    ],
})
display(references.style.hide(axis="index"))

In [ ]:
confidence = epi_result.conf_int()
epi_or = pd.DataFrame({
    "coeficiente (log-odds)": epi_result.params,
    "OR ajustada": np.exp(epi_result.params),
    "IC95% inferior": np.exp(confidence[0]),
    "IC95% superior": np.exp(confidence[1]),
    "valor-p": epi_result.pvalues,
}).drop(index="const")
epi_or.index.name = "atributo ou contraste (referência omitida)"
display(epi_or.round({
    "coeficiente (log-odds)": 3, "OR ajustada": 3,
    "IC95% inferior": 3, "IC95% superior": 3, "valor-p": 4,
}))

or_plot = (
    epi_or.assign(distancia_de_OR_1=np.abs(epi_or["coeficiente (log-odds)"]))
    .nlargest(15, "distancia_de_OR_1")
    .sort_values("OR ajustada")
)
y_positions = np.arange(len(or_plot))
plt.figure(figsize=(10, 7))
plt.errorbar(
    or_plot["OR ajustada"], y_positions,
    xerr=np.vstack([
        or_plot["OR ajustada"] - or_plot["IC95% inferior"],
        or_plot["IC95% superior"] - or_plot["OR ajustada"],
    ]),
    fmt="o", capsize=3,
)
plt.yticks(y_positions, or_plot.index)
plt.axvline(1, color="black", linestyle="--", linewidth=1)
plt.xscale("log")
plt.xlabel("Odds ratio ajustada (escala logarítmica) e IC95% robusto")
plt.title("Associações condicionais mais distantes de OR = 1")
plt.tight_layout()
plt.show()

### Como interpretar as odds ratios

`exp(coeficiente)` é a razão de *odds* associada ao contraste indicado, mantendo as demais variáveis constantes. OR = 1,25 representa *odds* 25% maiores; não significa probabilidade ou risco 25% maior. Se o IC95% inclui 1, os dados são compatíveis também com ausência de associação nesse nível de incerteza.

Os valores-p são exibidos para transparência, mas magnitude, intervalo de confiança, plausibilidade e pergunta científica devem orientar a leitura — não uma regra automática de significância.

### SHAP da regressão logística

SHAP complementa a OR: a OR resume um contraste global do modelo, enquanto SHAP mostra quanto o valor observado de cada atributo desloca uma previsão em relação à saída média. No modelo logístico linear, essas contribuições são aditivas em **log-odds**. Valores SHAP positivos aumentam a log-odds estimada da classe 1; valores negativos a reduzem.

In [ ]:
epi_features = epi_design.drop(columns="const")
epi_background = epi_features.sample(
    min(500, len(epi_features)), random_state=RANDOM_STATE
)
epi_masker = shap.maskers.Independent(
    epi_background, max_samples=len(epi_background)
)
epi_shap_sample = epi_features.sample(
    min(300, len(epi_features)), random_state=RANDOM_STATE + 1
)
epi_explainer = shap.LinearExplainer(
    (
        epi_result.params[epi_features.columns].to_numpy(),
        float(epi_result.params["const"]),
    ),
    epi_masker,
)
epi_shap_values = epi_explainer(epi_shap_sample)
shap.plots.bar(epi_shap_values, max_display=12)
shap.plots.beeswarm(epi_shap_values, max_display=12)

A OR e o SHAP descrevem o mesmo modelo por ângulos diferentes. Nenhum deles identifica efeito causal. Variáveis correlacionadas podem dividir ou deslocar contribuições, e as dummies devem ser interpretadas contra a categoria de referência omitida.

## Random Forest: avaliação preditiva no teste

> Como o segundo classificador se comporta em dados que não participaram do ajuste?

A Random Forest abaixo é um **segundo modelo**. Primeiro ela é ajustada somente no treinamento e avaliada no teste reservado. Assim como na regressão logística, não exibiremos desempenho no conjunto usado para o ajuste.

In [ ]:
forest_imputer = SimpleImputer(strategy="median")
X_train_i = pd.DataFrame(
    forest_imputer.fit_transform(X_train), columns=X.columns, index=X_train.index
)
X_test_i = pd.DataFrame(
    forest_imputer.transform(X_test), columns=X.columns, index=X_test.index
)
forest_evaluation = RandomForestClassifier(
    n_estimators=150,
    class_weight="balanced_subsample",
    n_jobs=-1,
    random_state=RANDOM_STATE,
)
forest_evaluation.fit(X_train_i, y_train)
forest_test_prob = forest_evaluation.predict_proba(X_test_i)[:, 1]
forest_test_pred = (forest_test_prob >= 0.5).astype(int)
forest_test_report = classification_report_health(
    y_test, forest_test_pred, forest_test_prob
)
display(report_frame(forest_test_report).round(3))
ConfusionMatrixDisplay.from_predictions(
    y_test, forest_test_pred,
    display_labels=[target_labels[0], target_labels[1]], cmap="Greens",
)
plt.title("Random Forest: matriz de confusão — teste")
plt.show()
plot_classification_curves(y_test, forest_test_prob)
plt.show()

### Como interpretar o desempenho

As medidas, a matriz de confusão e as curvas acima usam **somente o teste reservado**. Elas podem ser comparadas às da regressão logística porque os dois classificadores receberam a mesma divisão. Nenhuma métrica de desempenho do treinamento é mostrada.

### Importância por permutação no teste

A permutação usa a Random Forest ajustada no treinamento e o teste reservado. Assim, mede quanto cada atributo contribui para a capacidade de generalização observada, sem reutilizar os dados de ajuste.

In [ ]:
importance = permutation_importance(
    forest_evaluation, X_test_i, y_test,
    scoring="roc_auc", n_repeats=3,
    random_state=RANDOM_STATE, n_jobs=-1,
)
permutation = pd.Series(importance.importances_mean, index=X.columns).nlargest(12)
permutation.sort_values().plot.barh(
    title="Importância por permutação no teste (ROC-AUC)"
)
plt.xlabel("Queda média na ROC-AUC do teste")
plt.show()

### SHAP do modelo final reajustado

Depois de concluir a avaliação, a Random Forest e o imputador são reajustados na amostra completa. O SHAP abaixo descreve esse **modelo final**, que é diferente do modelo usado para obter as métricas e a importância por permutação no teste. Não calculamos novas medidas de desempenho na amostra completa.

In [ ]:
full_imputer = SimpleImputer(strategy="median")
X_full_i = pd.DataFrame(
    full_imputer.fit_transform(X), columns=X.columns, index=X.index
)
forest = RandomForestClassifier(
    n_estimators=150,
    class_weight="balanced_subsample",
    n_jobs=-1,
    random_state=RANDOM_STATE,
)
forest.fit(X_full_i, y)

shap_sample = X_full_i.sample(min(300, len(X_full_i)), random_state=RANDOM_STATE)
shap_background = X_full_i.sample(
    min(100, len(X_full_i)), random_state=RANDOM_STATE + 1
)
explainer = shap.Explainer(forest, shap_background)
shap_values = explainer(shap_sample)
if shap_values.values.ndim == 3:  # saída por classe em versões recentes
    shap_values = shap_values[..., 1]
shap.plots.bar(shap_values, max_display=12)
shap.plots.beeswarm(shap_values, max_display=12)

In [ ]:
mean_absolute_shap = np.abs(shap_values.values).mean(axis=0)
main_feature = shap_values.feature_names[int(np.argmax(mean_absolute_shap))]
shap.plots.scatter(shap_values[:, main_feature])

### Como interpretar a explicabilidade

A permutação descreve a dependência da ROC-AUC no teste em relação a cada atributo. O SHAP decompõe as saídas da Random Forest final, reajustada na amostra completa. As duas análises respondem a perguntas diferentes e nenhuma delas diz que mudar o atributo causará mudança de saúde.

## Limitações e responsabilidade

- O alvo e os atributos incluem autorrelato e não representam um diagnóstico produzido pelo notebook.
- Desbalanceamento, limiar, subgrupos e mudança de população alteram os erros.
- As OR são associações condicionais: ajustar todas as variáveis não substitui uma pergunta causal, um DAG e a definição prévia de confundidores.
- Os IC95% HC3 não incorporam pesos, estratos ou conglomerados do desenho amostral complexo do BRFSS; não são estimativas populacionais nacionais.
- O modelo supõe efeito linear na log-odds para IMC e dias de saúde ruim e não inclui interações; essas hipóteses precisam ser avaliadas em uma análise epidemiológica real.
- A importância por permutação no teste descreve a dependência do desempenho do modelo avaliado e pode subestimar atributos correlacionados.
- SHAP pode refletir correlações, vieses e atalhos do modelo; não é explicação causal.

## Atividade

Escolha uma exposição, interprete sua OR ajustada e IC95%, e compare essa leitura global com sua distribuição no gráfico SHAP. Depois teste limiares 0,3; 0,5; 0,7 no modelo preditivo e compare sensibilidade e especificidade.

## Três aprendizados principais

1. O pipeline protege o teste durante o ajuste; ROC-AUC não substitui matriz de confusão nem PR-AUC.
2. Desempenho e permutação da Random Forest usam o teste; a amostra completa é usada no GLM epidemiológico e no SHAP do modelo final reajustado.
3. OR e SHAP respondem a perguntas diferentes, a Random Forest é um modelo separado e nenhuma dessas explicações demonstra causalidade.

## Referências

- [UCI — CDC Diabetes Health Indicators](https://archive.ics.uci.edu/dataset/891/cdc+diabetes+health+indicators)
- [statsmodels — Generalized Linear Models](https://www.statsmodels.org/stable/glm.html)
- [SHAP — LinearExplainer](https://shap.readthedocs.io/en/stable/generated/shap.LinearExplainer.html)
- [scikit-learn — model evaluation](https://scikit-learn.org/stable/modules/model_evaluation.html)

## Versões das bibliotecas

Registre o ambiente junto ao resultado.

In [ ]:
from src.config import library_versions
library_versions(('numpy', 'pandas', 'scikit-learn', 'matplotlib', 'seaborn', 'shap', 'statsmodels', 'ucimlrepo'))